<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 3: Manufacturing Signals and<br>Feature Engineering</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week03/lab03_manufacturing_signals_and_feature_engineering.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

**ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science**  
**Wayne State University**

**Graded individual Lab · Approximately 90 minutes including submission**  
**Version:** Student Notebook

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and learning objectives

**Workflow:** raw signal -> time/features -> FFT/STFT -> feature table ->
relevance/redundancy -> three-feature subset -> guided ML and sound previews.
After this Lab you can:

- construct physical time and identify sensor units and cutter/cut grouping;
- compute mean, sample SD, RMS and peak-to-peak and interpret force/vibration differences;
- organize one cut's feature vector as a table row;
- interpret whole-record FFT amplitude and time-localized STFT PSD;
- use Pearson and redundancy evidence to justify an exploratory feature subset;
- separate predictors, target and groups for later regression/validation;
- explain a guided linear-frequency versus Mel manufacturing-sound example.

**Understand → apply → visualize → interpret.**
Most code is supplied. Complete seven short code entries and seven short responses.
FFT/STFT code is complete; run it and interpret. An error-free starter is not a
completed submission. Follow the notebook order; CP3 moves to Stage B after CP5.

**Time guide:** Stage A 40 min; Stage B 15 min; Stage C 15 min.
Guided previews 10 min; check and submit 10 min. **Estimated total: 90 min.**

Use NumPy, pandas, SciPy, Matplotlib and librosa. In Colab, run imports without
an installation cell. For local work, install the required packages in your local Python environment.
If a requirements.txt file is provided, you may use it to install those packages.
Submit Lab03_Firstname_Lastname.ipynb and .pdf through Canvas; no separate report.
Canvas controls deadlines. Guided/optional work is ungraded.

## Data and context
This Lab uses PHM 2010 milling data and MIMII manufacturing sound.
PHM: c1 cuts 1/158/315, Force X (N), Vibration X (g), sampling 50 kHz.
The master contains 945 cuts from c1/c4/c6. Cut numbers are sequence positions.
MIMII: Purohit et al. (2019), [dataset DOI](https://doi.org/10.5281/zenodo.3384388),
[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/); course-derived
mono Channel 1 audio. See the [PHM data card](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md)
and [MIMII data card](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/mimii/README.md)
for complete provenance, rights, attribution and transformations.

## Provided setup — run unchanged

These cells load libraries and data. File access and plotting syntax are not assessed.
Only edit cells marked REQUIRED; `None` means an answer has not yet been entered.

In [ ]:
# [GUIDED - RUN UNCHANGED]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq
from scipy.signal import spectrogram
from scipy.signal.windows import hann
from scipy.io import wavfile
import librosa
from IPython.display import display

In [ ]:
# [GUIDED - RUN UNCHANGED]
# These supplied settings keep every plot readable; no settings to edit.
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.precision', 4)  # Only displayed digits change.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Each URL is the address of one unchanged public course file.
BASE_URL = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/'
URL_CUT1 = BASE_URL + 'data/phm2010/c1_selected_cuts/c1_cut001.csv'
URL_CUT158 = BASE_URL + 'data/phm2010/c1_selected_cuts/c1_cut158.csv'
URL_CUT315 = BASE_URL + 'data/phm2010/c1_selected_cuts/c1_cut315.csv'
URL_MASTER = BASE_URL + 'data/phm2010/features/phm2010_features.csv'
URL_AUDIO = BASE_URL + 'data/mimii/audio/fan/id_00/normal/00000025.wav'

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Each DataFrame holds one complete cut: sample rows, sensor columns.
# round_trip preserves the precision stored in the CSV; leave it unchanged.
cut1 = pd.read_csv(URL_CUT1, float_precision='round_trip')
cut158 = pd.read_csv(URL_CUT158, float_precision='round_trip')
cut315 = pd.read_csv(URL_CUT315, float_precision='round_trip')
fs = 50_000  # Sampling rate: the sensors recorded 50,000 samples per second.

# Stage A — Understand One Manufacturing Signal

**40 minutes.** Time → scalars → frequency → time–frequency.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# np.arange(5) creates sample numbers 0, 1, 2, 3, 4.
# Dividing by samples/second gives elapsed time in seconds.
example_time_s = np.arange(5) / fs
print(example_time_s)

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Use cut 158 to connect the time axis and features to one real record.
force_x = cut158['force_x_N'].to_numpy()  # Force samples in newtons (N).
vibration_x = cut158['vibration_x_g'].to_numpy()  # Acceleration samples in g.
N = len(vibration_x)  # Number of samples in this complete record.
display(cut158.head(3))  # Show only three rows, not the entire recording.

One CSV row is one simultaneous sample of seven channels. For N samples,
`t[n] = n/fs`, nominal duration is N/fs, and the last sample is at (N-1)/fs.
At 50 kHz, the sample interval is 20 microseconds and Nyquist is 25 kHz.
The Nyquist limit is $f_{\mathrm{Nyquist}}=f_s/2$; components above it can alias
into lower frequencies. This is not evidence of an aliasing event in PHM.
The detailed synthetic comparison stays in lecture. Use `np.arange(N)` for sample indices.

## [REQUIRED CHECKPOINT 1] Physical time axis and labeled waveform

In [ ]:
# np.arange(N) gives sample numbers; dividing by fs gives seconds.
time_s = None  # TODO: Create the physical time axis using N and fs.
# Nominal recording duration in seconds; supplied.
duration_s = N / fs

In [ ]:
# [GUIDED - RUN UNCHANGED]
# fig is the whole figure; ax is its plotting area. Labels carry physical units.
if time_s is None:
    print('Complete the time-axis inputs in Checkpoint 1, then rerun.')
else:
    print('fs (Hz), N, duration (s), last sample (s):', fs, N, duration_s, time_s[-1])
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(time_s, vibration_x, linewidth=0.3)
    ax.set(title='c1 cut 158: Vibration X', xlabel='Time (s)', ylabel='Vibration X (g)')
    fig.tight_layout()
plt.show()

### Checkpoint 1 response (1–3 sentences)

What does a raw CSV row represent, and why is the last sample time before N/fs?

**TODO:** Write your response here.

## Four time-domain features

![One synthetic signal: mean, sample SD, RMS and peak-to-peak](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab03/four_features.png)

Synthetic conceptual example. Exact definitions follow; the SD band is not a probability guarantee.

**Mean.** Signed average level: the horizontal reference line.

$$\bar{x}=\frac{1}{N}\sum_{i=1}^{N}x_i$$

**Sample SD.** Fluctuation around the mean; use ddof=1.

$$s=\sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(x_i-\bar{x})^2}$$

**RMS.** Overall magnitude relative to zero, including the mean.

$$x_{\mathrm{RMS}}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}x_i^2}$$

**Peak-to-peak.** Full observed range; sensitive to extremes.

$$x_{p-p}=x_{\max}-x_{\min}$$

All retain the sensor unit. The exact finite-sample identity is
$$x_{\mathrm{RMS}}^2=\bar{x}^2+\frac{N-1}{N}s^2.$$
A nonzero force mean can make RMS exceed SD. Near-zero-mean vibration can
have almost equal RMS and SD. Do not center before raw RMS.

**Completed Force X example.** Each result is one number in N. Apply the same four calculations to Vibration X below.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Average signed force level across the complete cut (N).
force_mean = np.mean(force_x)
# Fluctuation around that mean; ddof=1 gives sample SD (N).
force_sd = np.std(force_x, ddof=1)
# Overall magnitude relative to zero, including mean load (N).
force_rms = np.sqrt(np.mean(force_x**2))
# Distance from smallest to largest observed force (N).
force_p2p = np.max(force_x) - np.min(force_x)
print(force_mean, force_sd, force_rms, force_p2p)

## [REQUIRED CHECKPOINT 2] Four features for Force X and Vibration X

In [ ]:
# Apply the Force X pattern to vibration_x; all four answers are in g.
vibration_mean = None  # TODO: Average the signed vibration samples.
vibration_sd = None  # TODO: Find fluctuation around the mean using sample SD.
vibration_rms = None  # TODO: Find overall magnitude relative to zero.
vibration_p2p = None  # TODO: Find the full observed range.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Lists keep the same order: mean, sample SD, RMS, peak-to-peak.
force_values = [force_mean, force_sd, force_rms, force_p2p]
vibration_values = [vibration_mean, vibration_sd, vibration_rms, vibration_p2p]
# Each list becomes one table row; column names explain its four numbers.
core_table = pd.DataFrame([force_values, vibration_values])
core_table.columns = ['Mean', 'Sample SD', 'RMS', 'Peak-to-peak']
core_table.index = ['Force X (N)', 'Vibration X (g)']
display(core_table)
if None in vibration_values:
    print('Complete the four Vibration X calculations in Checkpoint 2.')

### Checkpoint 2 response (1–3 sentences)

Explain the features physically and the difference between Force X and Vibration X SD/RMS.

**TODO:** Write your response here.

## FFT — which frequencies are present?

Run the completed code; interpretation is graded, not API memorization.

### Checkpoint 4 — Whole-record frequency content

In [ ]:
# [GUIDED - RUN UNCHANGED]
# vibration_late contains cut 315 acceleration samples in g.
vibration_late = cut315['vibration_x_g'].to_numpy()
# Remove the mean so the spectrum emphasizes oscillation.
oscillation = vibration_late - np.mean(vibration_late)
# window weights the record smoothly to reduce spectral leakage.
window = hann(len(oscillation), sym=False)
# spectrum holds the FFT coefficients of the windowed acceleration.
spectrum = rfft(oscillation * window)
# frequency_hz assigns a physical frequency to each FFT coefficient.
frequency_hz = rfftfreq(len(oscillation), d=1/fs)

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Supplied amplitude conversion: correct for the window, retain g units.
amplitude_g = np.abs(spectrum) / np.sum(window)
# Double positive-frequency amplitudes, excluding DC and any Nyquist endpoint.
# This slice is supplied normalization; you do not need to reproduce it.
amplitude_g[1:1+(len(oscillation)-1)//2] *= 2
peak = 1 + np.argmax(amplitude_g[1:])  # Index of the largest nonzero-frequency amplitude.
print('Non-DC peak (Hz, g):', frequency_hz[peak], amplitude_g[peak])
print('Bin spacing (Hz):', fs/len(oscillation))

In [ ]:
# [GUIDED - RUN UNCHANGED]
fig, ax = plt.subplots()
ax.plot(frequency_hz/1000, amplitude_g, linewidth=0.7)
ax.set(title='c1 cut 315: whole-record FFT', xlabel='Frequency (kHz)',
       ylabel='Amplitude (g)', xlim=(0, 25))
fig.tight_layout()
plt.show()

Bin spacing is not physical resolving power. Windowing spreads energy; off-bin amplitudes can be biased.

### Checkpoint 4 response (1–3 sentences)

What prominent component is present? What timing information is unavailable, and is the peak automatically a useful wear predictor?

**TODO:** Write your response here.

## STFT — when are frequencies present?

![Conceptual FFT whole-record spectrum versus overlapping-window STFT](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab03/fft_vs_stft.png)

Conceptual comparison of a whole-record FFT and windowed STFT. Illustrative panels are not matched PHM calculations.

### Checkpoint 5 — Time-localized frequency content

In [ ]:
# [GUIDED - RUN UNCHANGED]
nperseg = 2048  # Samples analyzed in one short window.
noverlap = 1536  # Samples shared by consecutive windows.
hop = 512  # Samples advanced between window starts: 2048 - 1536.
# SciPy returns frequency (Hz), window-center time (s), and PSD (g²/Hz).
# Keep the full-record centering; no additional per-window mean removal.
stft_hz, frame_s, psd = spectrogram(
    oscillation, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap,
    nfft=nperseg, detrend=False, scaling='density', mode='psd')
# db displays PSD on a logarithmic scale; the small floor prevents log(0).
db = 10 * np.log10(np.maximum(psd, 1e-20))

In [ ]:
# [GUIDED - RUN UNCHANGED]
fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.pcolormesh(frame_s, stft_hz/1000, db, shading='auto',
                     vmin=db.max()-80, vmax=db.max(), rasterized=True)
ax.set(title='c1 cut 315: STFT, Hann 2048 / hop 512', xlabel='Window-center time (s)',
       ylabel='Frequency (kHz)', ylim=(0, 25))
fig.colorbar(image, ax=ax, label='PSD (dB re 1 g²/Hz)')
fig.tight_layout()
plt.show()

Windows span 40.96 ms and advance 10.24 ms. SciPy uses complete frames without boundary padding here. PSD colors are not FFT amplitudes.

### Checkpoint 5 response (1–3 sentences)

What does FFT retain and lose, and what does STFT add? Describe one visible time-dependent pattern without assigning a cutting phase or cause.

**TODO:** Write your response here.

# Stage B — From Signals to a Dataset

**15 minutes.** One full cut becomes one row.

## [REQUIRED CHECKPOINT 3] One complete cut becomes one table row

**Provided reuse — run unchanged.** We package the same four calculations into a function so we do not rewrite them for every signal. `values` is a list in the order mean, SD, RMS, peak-to-peak. No function-writing task.

In [ ]:
# [GUIDED - RUN UNCHANGED]
def core_features(signal):
    # signal is one full sensor column; results retain that sensor's unit.
    mean_value = np.mean(signal)
    sd_value = np.std(signal, ddof=1)
    rms_value = np.sqrt(np.mean(signal**2))
    p2p_value = np.max(signal) - np.min(signal)
    values = [mean_value, sd_value, rms_value, p2p_value]
    return values

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Each call summarizes one whole force signal; all results are in N.
force1 = core_features(cut1['force_x_N'])
force158 = core_features(cut158['force_x_N'])
force315 = core_features(cut315['force_x_N'])
# One list becomes one cut's row. Labels are supplied, not computed dynamically.
force_table = pd.DataFrame([force1, force158, force315])
force_table.columns = ['Mean', 'Sample SD', 'RMS', 'Peak-to-peak']
force_table.index = [1, 158, 315]
force_table.index.name = 'Cut number'
print('Force X features (N)')
display(force_table)

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Each call summarizes one whole vibration signal; all results are in g.
vibration1 = core_features(cut1['vibration_x_g'])
vibration158 = core_features(cut158['vibration_x_g'])
vibration315 = core_features(cut315['vibration_x_g'])
# One list becomes one cut's row. Labels are supplied, not computed dynamically.
vibration_table = pd.DataFrame([vibration1, vibration158, vibration315])
vibration_table.columns = ['Mean', 'Sample SD', 'RMS', 'Peak-to-peak']
vibration_table.index = [1, 158, 315]
vibration_table.index.name = 'Cut number'
print('Vibration X features (g)')
display(vibration_table)

### Checkpoint 3 response (1–3 sentences)

Describe one change across the cuts. What is one feature-table row, and what cannot these three records establish?

**TODO:** Write your response here.

## From a feature vector to a table

![Raw signal to a feature vector, a cut-level table row and later ML inputs](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab03/signal_to_features.png)

**[GUIDED; CP3 continuation]** One raw record -> four summaries on two
channels -> an eight-feature vector -> one row with a cutter/cut key.
Revisit CP3; no duplicate table or answer. Windows from one record are not
independent wear experiments.

## Inspect the 945-cut dataset

In [ ]:
# [GUIDED - RUN UNCHANGED]
# master holds one row per cut, including sensor features and tool wear.
master = pd.read_csv(URL_MASTER, float_precision='round_trip')
# These five sensor columns keep our selection task physically inspectable.
candidate_features = [
    'force_x_mean',  # Average signed force (N).
    'force_x_sd',  # Force fluctuation (N).
    'vibration_x_sd',  # Acceleration fluctuation (g).
    'vibration_x_rms',  # Total acceleration magnitude (g).
    'ae_rms_mean',  # Average processed AE-RMS channel level (V).
]
print('Master rows and columns:', master.shape)
display(master[['cutter_id', 'cut_number', 'wear_mean_um']].head(3))

Use wear_mean_um as the continuous target. Wear level is categorical despite
integer storage; do not use it in Pearson correlation. Flute wear and wear_max_um
also leak target information if included as predictors. Cut number is sequence,
not a default predictor. AE-RMS mean is average processed signal level (V), and
AE-RMS SD is its fluctuation (V); neither is a raw AE frequency measurement.

# Stage C — Which Features Should We Use?

**15 minutes.** Combine relevance, redundancy and engineering meaning.

For paired feature values $x_i$ and mean wear $y_i$ across cuts:
$$r_{xy}=\frac{\sum_{i=1}^{N}(x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum_{i=1}^{N}(x_i-\bar{x})^2}\sqrt{\sum_{i=1}^{N}(y_i-\bar{y})^2}}.$$

- $r$ describes the strength and direction of a linear association.
- Correlation does not imply causation or guaranteed predictive value.

## [REQUIRED CHECKPOINT 6] Feature–wear relationship and Pearson correlation

Run this supplied plot for **force_x_mean versus wear_mean_um**. Then
complete one Pearson calculation and interpret the plot and r together.
Pearson measures linear association, not causation. Low r does not rule out
nonlinear or cutter-dependent information.

**Provided cutter comparison.** The condition inside brackets keeps only rows for that cutter. The three colors let us see whether the pooled trend hides cutter differences.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# c1, c4 and c6 retain the same columns but contain different cutter rows.
c1 = master[master['cutter_id'] == 'c1']
c4 = master[master['cutter_id'] == 'c4']
c6 = master[master['cutter_id'] == 'c6']
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(c1['wear_mean_um'], c1['force_x_mean'], s=8, label='c1')
ax.scatter(c4['wear_mean_um'], c4['force_x_mean'], s=8, label='c4')
ax.scatter(c6['wear_mean_um'], c6['force_x_mean'], s=8, label='c6')
ax.set(xlabel='Mean wear (µm)', ylabel='Force X mean (N)',
       title='Force X mean versus wear, by cutter')
ax.legend()
plt.show()

**Syntax:** `table['feature'].corr(table['target'], method='pearson')` returns one r.

In [ ]:
# Series.corr() compares two columns and returns one Pearson r value.
# target_r describes linear association with wear, not causation.
target_r = None  # TODO: Compare the Force X mean and mean-wear columns using .corr().
print('Pearson r:', target_r)

### Checkpoint 6 response (1–2 sentences)

Describe the force–wear association shown by the scatter plot and r. State one limitation of this pooled relationship.

**TODO:** Write your response here.

**Selection recap:** relevance to wear + redundancy between features + engineering meaning.
Filter/wrapper/embedded taxonomy stays in lecture; no model is trained here.

## [REQUIRED CHECKPOINT 7] Identify redundancy and select three features

Run the supplied summary and heatmap. Inspect one redundant pair, choose
**three** features, and justify the list in 2–3 sentences. No correlation-matrix
coding, ranking or significance tests are required. This is a simple filter-style
exercise; wrapper/embedded methods remain lecture context.

**Guided comparison — run unchanged.** First compare each candidate with wear. Then compare Force X mean within each cutter; a pooled correlation need not describe each cutter.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Each r value describes one feature's association with wear (no unit).
force_mean_r = master['force_x_mean'].corr(master['wear_mean_um'])
force_sd_r = master['force_x_sd'].corr(master['wear_mean_um'])
vibration_sd_r = master['vibration_x_sd'].corr(master['wear_mean_um'])
vibration_rms_r = master['vibration_x_rms'].corr(master['wear_mean_um'])
ae_r = master['ae_rms_mean'].corr(master['wear_mean_um'])
relevance = pd.DataFrame()
relevance['Feature'] = candidate_features
relevance['Pearson r with wear'] = [force_mean_r, force_sd_r, vibration_sd_r, vibration_rms_r, ae_r]
display(relevance)

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Repeat the same Pearson expression using each cutter's rows.
c1_r = c1['force_x_mean'].corr(c1['wear_mean_um'])
c4_r = c4['force_x_mean'].corr(c4['wear_mean_um'])
c6_r = c6['force_x_mean'].corr(c6['wear_mean_um'])
cutter_correlations = pd.DataFrame()
cutter_correlations['Cutter'] = ['c1', 'c4', 'c6']
cutter_correlations['Pearson r'] = [c1_r, c4_r, c6_r]
display(cutter_correlations)

**[GUIDED — RUN UNCHANGED]** Rows and columns are the five candidate features. The supplied loops visit each row/column intersection to print its number; do not edit the plotting code.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# Values near +1 or -1 indicate a strong linear relationship.
# Large feature-feature correlation can indicate redundant information.
feature_corr = master[candidate_features].corr(method='pearson')
fig, ax = plt.subplots(figsize=(9, 6))
image = ax.imshow(feature_corr, vmin=-1, vmax=1, cmap='coolwarm')
labels = ['Force mean', 'Force SD', 'Vibration SD', 'Vibration RMS', 'AE-RMS mean']
ax.set_xticks(range(5), labels, rotation=30, ha='right')
ax.set_yticks(range(5), labels)
ax.set_title('Five candidates: Pearson correlation')
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{feature_corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=12)
fig.colorbar(image, ax=ax, label='Pearson r (dimensionless)')
fig.tight_layout()
plt.show()

In [ ]:
# [GUIDED - RUN UNCHANGED]
# For near-zero-mean vibration, SD and RMS can carry nearly redundant information.
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(c1['vibration_x_sd'], c1['vibration_x_rms'], s=8, label='c1')
ax.scatter(c4['vibration_x_sd'], c4['vibration_x_rms'], s=8, label='c4')
ax.scatter(c6['vibration_x_sd'], c6['vibration_x_rms'], s=8, label='c6')
ax.set(xlabel='Vibration X sample SD (g)', ylabel='Vibration X raw RMS (g)',
       title='Redundancy check: SD versus RMS')
ax.legend()
fig.tight_layout()
plt.show()

Choose exactly **three distinct names from candidate_features**. Consider relevance,
redundancy and physical meaning; there is no unique correct subset.

**This is exploratory feature selection, not an optimized feature-selection
procedure.** The 945 cuts are three related trajectories. Full-data inspection
does not establish independent test performance; later data-dependent selection
and preprocessing must use training data only.

In [ ]:
# Choose three names from candidate_features.
# Justify them using relevance, redundancy and physical meaning.
selected_features = None  # TODO: Replace None with a list of exactly three candidate names.
print('Selected:', selected_features)

### Checkpoint 7 response (2–3 sentences)

Identify a redundant pair and justify your retained/omitted features using relevance, redundancy and engineering meaning. State the exploratory limitation.

**TODO:** Write your response here.

## Guided preview — X, y and groups

**Ungraded; included in the 10-minute guided preview.**

**[GUIDED; UNGRADED]** Run unchanged after selecting three features; inspect
the shapes and columns. X and y preview Week 4 regression; cutter groups preview
Week 6 group-aware validation. No extra answer, splitting or model training.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# None means Checkpoint 7 is unfinished; skip this preview until ready.
if selected_features is None:
    print('Choose three feature names in Checkpoint 7, then rerun.')
else:
    # X keeps the three selected sensor columns; each row is a cut.
    X = master[selected_features]
    # y is mean tool wear (µm), the Week 4 regression target.
    y = master['wear_mean_um']
    # groups stores the cutter for each row, for Week 6 validation.
    groups = master['cutter_id']
    print('X / y / groups shapes:', X.shape, y.shape, groups.shape)

# Preview — Manufacturing Sound

**Ungraded; included in the 10-minute guided preview.** Run the supplied library calls and inspect four views.

**1. Load the WAV — run unchanged.** The supplied download lines hand the file to SciPy. Integer PCM values are converted to a fraction of digital full scale (FS), not sound pressure.

In [ ]:
# [GUIDED - RUN UNCHANGED]
from io import BytesIO
from urllib.request import urlopen
# audio_fs is samples/second; pcm contains the recorded integer samples.
audio_fs, pcm = wavfile.read(BytesIO(urlopen(URL_AUDIO).read()))
# Divide by the fixed PCM16 scale; do not normalize to this recording's peak.
audio = pcm.astype(float) / 32768
audio_time = np.arange(len(audio)) / audio_fs  # Sample times in seconds.

**2. Waveform — run unchanged.** Plot digital amplitude against physical time.

In [ ]:
# [GUIDED - RUN UNCHANGED]
fig, ax = plt.subplots()
ax.plot(audio_time, audio, linewidth=0.3)
ax.set(title='Normal fan: full waveform', xlabel='Time (s)', ylabel='Amplitude (FS)')
fig.tight_layout(); plt.show()

**3. Linear-frequency spectrogram — run unchanged.** Reuse short overlapping windows; colors show power density over time.

In [ ]:
# [GUIDED - RUN UNCHANGED]
n_fft = 1024  # Audio samples in each FFT window.
hop_length = 256  # Samples between consecutive window starts.
n_mels = 64  # Mel bands used in the next view.
# Returned arrays: frequency in Hz, center time in s, PSD in FS²/Hz.
audio_hz, audio_s, audio_psd = spectrogram(
    audio - np.mean(audio), fs=audio_fs, window='hann', nperseg=n_fft,
    noverlap=n_fft-hop_length, nfft=n_fft, detrend=False, scaling='density', mode='psd')
# audio_db is a logarithmic display of PSD referenced to 1 FS²/Hz.
audio_db = librosa.power_to_db(audio_psd, ref=1.0, amin=1e-20, top_db=80)

In [ ]:
# [GUIDED - RUN UNCHANGED]
fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.pcolormesh(audio_s, audio_hz/1000, audio_db, shading='auto', rasterized=True)
ax.set(title='Fan: linear-frequency STFT', xlabel='Window-center time (s)',
       ylabel='Frequency (kHz)', ylim=(0, 8))
fig.colorbar(image, ax=ax, label='PSD (dB re 1 FS²/Hz)')
fig.tight_layout(); plt.show()

Linear frequency uses Hz; Mel uses auditory-spaced bands, common for sound. Mel colors show relative FFT-based power, not calibrated PSD. This representation is not automatically appropriate for force or vibration.

**4. Mel-spectrogram — run unchanged.** librosa combines frequency bins into auditory-spaced bands; this is a sound representation, not a new prediction model.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# mel_power has 64 band rows and one column per time window.
# power=2 uses squared FFT magnitude; 0–8000 Hz covers the audio Nyquist range.
# center=False keeps complete windows; htk=True sets the Mel spacing convention.
# norm=None retains unnormalized band weights; these are not calibrated PSD units.
mel_power = librosa.feature.melspectrogram(
    y=audio - np.mean(audio), sr=audio_fs, n_fft=n_fft, hop_length=hop_length,
    n_mels=n_mels, fmin=0, fmax=8000, power=2.0, window='hann',
    center=False, htk=True, norm=None)
# One center time (s) per column: start sample plus half a window.
mel_time = (np.arange(mel_power.shape[1])*hop_length + n_fft/2) / audio_fs

In [ ]:
# [GUIDED - RUN UNCHANGED]
fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.pcolormesh(mel_time, np.arange(64), mel_power, shading='auto', rasterized=True)
ax.set(title='Fan: Mel-spectrogram', xlabel='Window-center time (s)', ylabel='Mel band index')
fig.colorbar(image, ax=ax, label='Relative spectral power (FFT-based)')
fig.tight_layout()
plt.show()

**5. Convert Mel power to dB — run unchanged.** Logarithmic colors reveal weaker patterns; 0 dB is the largest value in this recording, not an absolute acoustic level.

In [ ]:
# [GUIDED - RUN UNCHANGED]
# mel_db retains the same rows/columns; only the color scale changes.
# ref=np.max sets the maximum to 0 dB; top_db limits the display to 80 dB below it.
mel_db = librosa.power_to_db(mel_power, ref=np.max, amin=1e-20, top_db=80)

In [ ]:
# [GUIDED - RUN UNCHANGED]
fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.pcolormesh(mel_time, np.arange(64), mel_db, shading='auto', rasterized=True)
ax.set(title='Fan: log-Mel-spectrogram', xlabel='Window-center time (s)', ylabel='Mel band index')
fig.colorbar(image, ax=ax, label='Power (dB relative to maximum)')
fig.tight_layout()
plt.show()

### Guided interpretation (1–3 sentences; ungraded)

How do linear-frequency and Mel representations differ? Why is Mel common for
sound, and why is it not automatically appropriate for force/vibration?

**TODO:** Write a short comparison.

# Optional Challenge — only if time permits

Ungraded: compare another PHM channel, Pearson versus Spearman, or another MIMII machine type. No model fitting or splits.

# Submission checklist

Rendered notebook checkboxes are not clickable. To record completion, edit this
Markdown cell and change `[ ]` to `[x]`.

- [ ] Student Information uses name and WSU AccessID.
- [ ] Seven checkpoints contain completed code, required outputs and short responses.
- [ ] CP1 waveform; CP2 feature table; CP3 three-cut table; CP4 FFT; CP5 STFT are visible.
- [ ] CP6 one scatter plot, one Pearson r and short interpretation are visible.
- [ ] CP7 supplied heatmap/redundancy plot, three-feature list and justification are visible.
- [ ] Run the ungraded X/y/groups preview and inspect its shapes; no extra answer.
- [ ] Guided sound example and short comparison are complete; optional work is not needed.
- [ ] Restart runtime/kernel and Run all after completing placeholders; resolve errors
  and all unfinished-checkpoint reminders. Save outputs without printing full raw data.
- [ ] Export a PDF and inspect code, tables, figures and responses for clipped content.
- [ ] Submit Lab03_Firstname_Lastname.ipynb and Lab03_Firstname_Lastname.pdf through Canvas.

No separate report. Canvas controls deadlines and policies.

# References

- [PHM Society 2010 challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/)
- [Course PHM provenance and license discussion](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md)
- [PHM feature dictionary and definitions](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md)
- Purohit et al., *MIMII Dataset: Sound Dataset for Malfunctioning Industrial Machine Investigation and Inspection*, 2019: [paper](https://arxiv.org/abs/1909.09347), [dataset](https://doi.org/10.5281/zenodo.3384388), [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- [Course MIMII transformations and metadata](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/mimii/README.md)
- [SciPy spectrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.spectrogram.html)
- [librosa Mel-spectrogram](https://librosa.org/doc/0.11.0/generated/librosa.feature.melspectrogram.html)

The course-derived scalar and Mel representations are instructional choices,
not claims about official benchmark protocols.